## Exercise 5 · A complete CSV cleaning pipeline

Build a reusable workflow with four design patterns:

| Pattern | Responsibility |
| :--- | :--- |
| **Template Method** | `DataPipeline.run()` fixes the order: load → clean → transform → save. |
| **Strategy** | `DataCleaner` delegates missing-value handling to an interchangeable strategy. |
| **Factory** | `TransformFactory` creates transformations from configuration names. |
| **Decorator** | `@log_action` and `@log_time` provide optional execution diagnostics. |

Run all cells in order. The example writes `dirty_sample.csv` and `cleaned_sample.csv`
in the notebook's working directory. Verbose diagnostics are disabled by default.

### 1. Imports and execution decorators

Set `ENABLE_LOGGING = True` to inspect calls, failures, and elapsed time.
Both decorators preserve the wrapped function's metadata; timing is reported even
when a call fails.

In [1]:
from abc import ABC, abstractmethod
from functools import wraps
from pathlib import Path
from time import perf_counter

import pandas as pd
from IPython.display import display

ENABLE_LOGGING = False


def log_action(func):
    """Optionally report the start, success, or failure of a call."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        if ENABLE_LOGGING:
            print(f"[LOG] {func.__qualname__} started")
        try:
            result = func(*args, **kwargs)
        except Exception as exc:
            if ENABLE_LOGGING:
                print(f"[LOG] {func.__qualname__} failed: {exc}")
            raise
        if ENABLE_LOGGING:
            print(f"[LOG] {func.__qualname__} finished")
        return result
    return wrapper


def log_time(func):
    """Optionally report elapsed time, including unsuccessful calls."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        if not ENABLE_LOGGING:
            return func(*args, **kwargs)
        start = perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            print(f"[TIME] {func.__qualname__}: {perf_counter() - start:.4f}s")
    return wrapper

### 2. CSV reader

Encapsulate CSV ingestion and provide a notebook-friendly preview.

In [2]:
class CSVReader:
    def __init__(self, file_path: str | Path):
        self.file_path = Path(file_path)
        self.df: pd.DataFrame | None = None

    @log_action
    @log_time
    def read(self) -> pd.DataFrame:
        self.df = pd.read_csv(self.file_path)
        return self.df

    def preview(self, n: int = 5) -> None:
        if self.df is None:
            raise RuntimeError("Call read() before preview().")
        display(self.df.head(n))

### 3. Missing-value strategies · Strategy Pattern

- **`DropMissing`** removes rows containing any missing cell.
- **`FillMean`** fills numeric columns with their means and preserves missing text.
- **`FillMode`** fills each column with its first mode; ties follow pandas' mode ordering.

All strategies return a new DataFrame. A column containing only missing values has
no mean or mode, so it remains missing. Empty or whitespace-only strings are not
treated as missing by these strategies.

In [3]:
class MissingValueStrategy(ABC):
    @abstractmethod
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Return cleaned data without changing the input DataFrame."""
        ...


class DropMissing(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.dropna().reset_index(drop=True)


class FillMean(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.fillna(df.mean(numeric_only=True))


class FillMode(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        # Only calculate modes for columns that need imputation.
        replacements = {}
        for name in df.columns[df.isna().any()]:
            modes = df[name].mode()
            if not modes.empty:
                replacements[name] = modes.iloc[0]
        return df.fillna(replacements)

### 4. Cleaning context

`DataCleaner` delegates to the selected strategy. With no strategy, it leaves the
data unchanged and avoids allocating an unnecessary copy.

In [4]:
class DataCleaner:
    def __init__(self, strategy: MissingValueStrategy | None = None):
        self.strategy = strategy

    @log_action
    @log_time
    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        return df if self.strategy is None else self.strategy.handle(df)

### 5. Reusable transformations

- **`NormalizeColumns`** strips headers, lowercases them, and collapses whitespace
  to underscores. It rejects headers that would become ambiguous duplicates.
- **`StandardizeText`** strips and lowercases text while preserving missing values
  and non-string values in mixed object columns.
- **`RemoveDuplicates`** removes identical rows and resets the index.

Order matters: standardize text before removing duplicates so cosmetic differences
do not hide duplicate records. Each transformation preserves its input.

In [5]:
class DataTransform(ABC):
    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        """Return transformed data without changing the input DataFrame."""
        ...


class NormalizeColumns(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        columns = (pd.Index(df.columns).astype(str).str.strip().str.lower()
                   .str.replace(r"\s+", "_", regex=True))
        if columns.has_duplicates:
            duplicates = columns[columns.duplicated()].unique().tolist()
            raise ValueError(f"Column normalization creates duplicate names: {duplicates}")
        return df.set_axis(columns, axis="columns")


class RemoveDuplicates(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.drop_duplicates().reset_index(drop=True)


class StandardizeText(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy()
        for name in result.select_dtypes(include=["object", "string"]).columns:
            column = result[name]
            if pd.api.types.infer_dtype(column, skipna=True) in {"string", "empty"}:
                result[name] = column.str.strip().str.lower()
            else:
                # The .str accessor alone would replace non-string values with NaN.
                result[name] = column.map(
                    lambda value: value.strip().lower() if isinstance(value, str) else value
                )
        return result

### 6. Transformation factory · Factory Pattern

Create the requested transformation from its name. Unknown names raise a clear error.

In [6]:
class TransformFactory:
    _transforms = {
        "normalize_columns": NormalizeColumns,
        "standardize_text": StandardizeText,
        "remove_duplicates": RemoveDuplicates,
    }

    @classmethod
    def get_transform(cls, name: str) -> DataTransform:
        key = name.strip().lower()
        if key not in cls._transforms:
            available = ", ".join(cls._transforms)
            raise ValueError(f"Unknown transform {name!r}. Available: {available}")
        return cls._transforms[key]()

### 7. Pipeline template · Template Method Pattern

The base class defines the workflow once. Concrete subclasses implement the four
steps. `run()` returns the pipeline so callers can inspect its result and summary.

In [7]:
class DataPipeline(ABC):
    @log_action
    @log_time
    def run(self):
        """Execute the fixed load → clean → transform → save sequence."""
        self.load()
        self.clean()
        self.transform()
        self.save()
        return self

    @abstractmethod
    def load(self) -> None:
        ...

    @abstractmethod
    def clean(self) -> None:
        ...

    @abstractmethod
    def transform(self) -> None:
        ...

    @abstractmethod
    def save(self) -> None:
        ...

### 8. Concrete CSV pipeline

Implement the four template steps using the reader, cleaner, and transformation factory.

In [8]:
class CSVDataPipeline(DataPipeline):
    def __init__(self, input_path, output_path, missing_strategy=None, transforms=None):
        self.input_path = input_path
        self.output_path = output_path
        self.cleaner = DataCleaner(missing_strategy)
        self.transforms = tuple(transforms) if transforms is not None else ()
        self.df = None

    def load(self):
        self.df = CSVReader(self.input_path).read()

    def clean(self):
        self.df = self.cleaner.clean(self.df)

    def transform(self):
        for name in self.transforms:
            transformer = TransformFactory.get_transform(name)
            self.df = transformer.apply(self.df)

    def save(self):
        self.df.to_csv(self.output_path, index=False)

### 9. Create the sample dataset

The five input rows contain inconsistent headers and text casing, one duplicate
Alice record, and three missing cells: age, salary, and department.

In [9]:
INPUT_PATH = Path("dirty_sample.csv")
OUTPUT_PATH = Path("cleaned_sample.csv")

raw_df = pd.DataFrame({
    " Full Name ": [" Alice ", "BOB", "Charlie ", " Alice ", "David"],
    " Age ": [25.0, None, 30.0, 25.0, 40.0],
    " Salary ": [50000.0, 60000.0, None, 50000.0, 75000.0],
    " Department ": [" HR ", "engineering", "FINANCE", " HR ", None],
})
raw_df.to_csv(INPUT_PATH, index=False)

### 10. Run the pipeline

Run the four steps and compare the original and cleaned data using standard pandas tables.

`FillMean` fills age with **30** and salary with **58,750** before deduplication.
David's department remains missing because it is not numeric.

In [10]:
pipeline = CSVDataPipeline(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    missing_strategy=FillMean(),
    transforms=["normalize_columns", "standardize_text", "remove_duplicates"],
).run()

cleaned_df = pipeline.df
print("Before cleaning:")
display(raw_df)
print("After cleaning:")
display(cleaned_df)

Before cleaning:


,Full Name,Age,Salary,Department
0,Alice,25.0,50000.0,HR
1,BOB,NaN,60000.0,engineering
2,Charlie,30.0,NaN,FINANCE
3,Alice,25.0,50000.0,HR
4,David,40.0,75000.0,NaN


After cleaning:


,full_name,age,salary,department
0,alice,25.0,50000.0,hr
1,bob,30.0,60000.0,engineering
2,charlie,30.0,58750.0,finance
3,david,40.0,75000.0,NaN
